# EEG Dataset Unit Check & Loader Test
KL 2026-01-06

In [1]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from loader import load_dataset, split_dataset, SplitStrategy
from hdf5_io import HDF5Reader
import numpy as np
import pandas as pd

In [17]:
import mne
file_path = '/mnt/dataset2/Datasets/Physionet_MI/files/eegmmidb/1.0.0/S001/S001R04.edf'
raw = mne.io.read_raw_edf(file_path, preload=True)
print(raw.annotations)

Extracting EDF parameters from /mnt/dataset2/Datasets/Physionet_MI/files/eegmmidb/1.0.0/S001/S001R04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
<Annotations | 30 segments: T0 (15), T1 (8), T2 (7)>


In [2]:
def detect_unit(data: np.ndarray) -> tuple[str, float]:
    """Detect data unit based on amplitude range."""
    max_amp = np.abs(data).max()
    if max_amp > 1e1:  # > 10, likely µV
        return "µV", max_amp
    return "V", max_amp


def check_dataset_unit(data_dir: str) -> dict:
    """Check unit and amplitude statistics for a dataset."""
    # Match both sub_*.h5 (underscore) and sub-*.h5 (hyphen) formats
    h5_files = sorted(list(Path(data_dir).glob("sub_*.h5")) + list(Path(data_dir).glob("sub-*.h5")))
    print(f"Found {len(h5_files)} HDF5 files in {data_dir}")
    if not h5_files:
        return {"error": "No HDF5 files found"}

    stats = {
        "dataset_name": None,
        "num_files": len(h5_files),
        "num_labels": None,
        "category_list": None,
        "detected_unit": None,
        "max_amplitude": 0,
        "min_amplitude": float('inf'),
        "mean_amplitude": [],
        "samples_checked": 0,
        "amplitude_violations": 0,
    }

    for h5_file in h5_files[:3]:
        print(f"Checking file: {h5_file}")
        with HDF5Reader(str(h5_file)) as reader:
            subj = reader.subject_attrs
            print(subj)
            if stats["dataset_name"] is None:
                stats["dataset_name"] = subj.dataset_name
                stats["num_labels"] = getattr(subj, 'num_labels', None)
                stats["category_list"] = getattr(subj, 'category_list', None)

            trial_names = reader.get_trial_names()[:3]
            for trial_name in trial_names:
                
                seg_names = reader.get_segment_names(trial_name)[:5]
                # `subj = reader.subject_attrs` is assigning the attribute `subject_attrs` of the `reader`
                # object to the variable `subj`. This line of code is retrieving the subject attributes from
                # the HDF5 file being read by the `reader` object. These attributes typically contain
                # information about the dataset, task type, subject ID, sampling rate, montage, channels, and
                # other relevant metadata related to the EEG data being processed.
                for seg_name in seg_names:
                    seg = reader.get_segment(trial_name, seg_name)
                    print(seg)
                    data = seg.data
                    print(data.shape)
                    
                    max_amp = np.abs(data).max()
                    
                    stats["max_amplitude"] = max(stats["max_amplitude"], max_amp)
                    stats["min_amplitude"] = min(stats["min_amplitude"], max_amp)
                    stats["mean_amplitude"].append(np.abs(data).mean())
                    stats["samples_checked"] += 1
                    print(f"  Segment: {seg_name}, Max Amplitude: {max_amp:.2f}, mean Amplitude: {np.abs(data).mean():.2f}, median Amplitude: {np.median(np.abs(data)):.2f}")
                    if max_amp > 600:
                        stats["amplitude_violations"] += 1

    stats["detected_unit"], _ = detect_unit(np.array([stats["max_amplitude"]]))
    stats["mean_amplitude"] = np.mean(stats["mean_amplitude"]) if stats["mean_amplitude"] else 0
    return stats

## Single Dataset Report

In [3]:
# Change this to your dataset path
# data_dir = './hdf5/HBN_EEG'
# data_dir = "/mnt/dataset2/Processed_datasets/EEG_Bench/SHU-MI"
# data_dir = '/mnt/dataset2/hdf5_datasets/TUEV'
# data_dir = '/mnt/dataset2/chr/full_version/output_tuh_full/TUH_EEG_FULL'
# data_dir = '/mnt/dataset2/benchmark_dataloader/hdf5/downstream/TUAB'
# data_dir = '/mnt/dataset2/benchmark_dataloader/test_output/EEG-IO'
# data_dir = "/mnt/dataset2/Processed_datasets/BCIC2A/BCIC2A"
data_dir = "/mnt/dataset2/Processed_datasets/EEG_Bench/BCIC2A"
# data_dir = '/mnt/dataset2/hdf5_datasets/TUEV'
# data_dir = '/mnt/dataset2/Processed_datasets/EEG_Pre/TUH'

In [2]:
import os 

import mne
file_path = '/mnt/dataset2/Datasets/Efficient dual-frequency SSVEP BCI system/Data_of_40targets_offline/Data of 40targets offline/Subject 01.cnt'
raw = mne.io.read_raw_cnt(file_path, preload=True)


Reading 0 ... 1732839  =      0.000 ...  1732.839 secs...


/tmp/ipykernel_834345/3441904974.py:5: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(file_path, preload=True)


In [5]:
df = raw.annotations.to_data_frame()

In [7]:
raw.annotations.count()

{'1': 8,
 '10': 8,
 '11': 8,
 '12': 8,
 '13': 8,
 '14': 8,
 '15': 8,
 '16': 8,
 '17': 8,
 '18': 8,
 '19': 8,
 '2': 8,
 '20': 8,
 '21': 8,
 '22': 8,
 '23': 8,
 '24': 8,
 '25': 8,
 '253': 320,
 '26': 8,
 '27': 8,
 '28': 8,
 '29': 8,
 '3': 8,
 '30': 8,
 '31': 8,
 '32': 8,
 '33': 8,
 '34': 8,
 '35': 8,
 '36': 8,
 '37': 8,
 '38': 8,
 '39': 8,
 '4': 8,
 '40': 8,
 '5': 8,
 '6': 8,
 '7': 8,
 '8': 8,
 '9': 8}

In [6]:
df

,onset,duration,description
0,1970-01-01 00:00:24.048,0.0,8
1,1970-01-01 00:00:26.064,0.0,253
2,1970-01-01 00:00:28.081,0.0,17
3,1970-01-01 00:00:30.097,0.0,253
4,1970-01-01 00:00:32.115,0.0,9
...,...,...,...
635,1970-01-01 00:28:40.147,0.0,253
636,1970-01-01 00:28:42.164,0.0,21
637,1970-01-01 00:28:44.181,0.0,253
638,1970-01-01 00:28:46.198,0.0,8


In [4]:
stats = check_dataset_unit(data_dir)
print(f"Dataset: {stats['dataset_name']}")
print(f"Files: {stats['num_files']}")
print(f"Samples checked: {stats['samples_checked']}")
print()
print("--- Label Info ---")
print(f"Num labels: {stats['num_labels']}")
print(f"Categories: {stats['category_list']}")
print()
print("--- Unit Analysis ---")
print(f"Detected unit: {stats['detected_unit']}")
print(f"Max amplitude: {stats['max_amplitude']:.4e}")
print(f"Min amplitude: {stats['min_amplitude']:.4e}")
print(f"Mean amplitude: {stats['mean_amplitude']:.4e}")
print()
print("--- Validation ---")
is_uv = stats['detected_unit'] == 'µV'
has_labels = stats['num_labels'] is not None and stats['num_labels'] > 0
has_categories = stats['category_list'] is not None and len(stats['category_list']) > 0
print(f"Unit is µV: {'✓' if is_uv else '✗ (NEEDS UPDATE)'}")
print(f"Has num_labels: {'✓' if has_labels else '✗ (NEEDS UPDATE)'}")
print(f"Has category_list: {'✓' if has_categories else '✗ (NEEDS UPDATE)'}")
if is_uv:
    print(f"Amplitude violations (>600 µV): {stats['amplitude_violations']}/{stats['samples_checked']}")
    


Found 9 HDF5 files in /mnt/dataset2/Processed_datasets/EEG_Bench/BCIC2A
Checking file: /mnt/dataset2/Processed_datasets/EEG_Bench/BCIC2A/sub_1.h5
SubjectAttrs(subject_id=1, dataset_name='BCIC2A_4class', task_type='motor_imaginary', downstream_task_type='classification', rsFreq=200.0, chn_name=['FZ', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4', 'P1', 'PZ', 'P2', 'POZ'], num_labels=4, category_list=['left', 'right', 'foot', 'tongue'], chn_pos=None, chn_ori=None, chn_type='EEG', montage='10_20', report='', clinical_metadata={})
EEGSegment(data=array([[ -6.820768  , -11.206616  , -13.457066  , ...,   6.492717  ,
          2.434435  ,  -0.8243348 ],
       [ -8.195013  , -10.501629  , -11.246537  , ...,   6.674732  ,
          2.8952098 ,   1.385365  ],
       [ -7.8250794 , -10.691619  , -11.754739  , ...,   8.321767  ,
          5.649194  ,   2.822359  ],
       ...,
       [-17.915777  , -15.703002  , -15.009149  , ...,  

## testing EEGIO dataset

In [11]:
import os
os.listdir('/mnt/dataset2/Datasets/HBN-EEG/ds005505-download/sub-NDARAC904DMU/eeg/')

['sub-NDARAC904DMU_task-DespicableMe_channels.tsv',
 'sub-NDARAC904DMU_task-DespicableMe_eeg.json',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_channels.tsv',
 'sub-NDARAC904DMU_task-DespicableMe_events.tsv',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_eeg.json',
 'sub-NDARAC904DMU_task-FunwithFractals_channels.tsv',
 'sub-NDARAC904DMU_task-FunwithFractals_eeg.json',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_events.tsv',
 'sub-NDARAC904DMU_task-RestingState_channels.tsv',
 'sub-NDARAC904DMU_task-FunwithFractals_events.tsv',
 'sub-NDARAC904DMU_task-DespicableMe_eeg.set',
 'sub-NDARAC904DMU_task-DiaryOfAWimpyKid_eeg.set',
 'sub-NDARAC904DMU_task-FunwithFractals_eeg.set',
 'sub-NDARAC904DMU_task-RestingState_eeg.json',
 'sub-NDARAC904DMU_task-RestingState_events.tsv',
 'sub-NDARAC904DMU_task-ThePresent_channels.tsv',
 'sub-NDARAC904DMU_task-ThePresent_eeg.json',
 'sub-NDARAC904DMU_task-ThePresent_events.tsv',
 'sub-NDARAC904DMU_task-RestingState_eeg.set',
 'sub-NDARAC904DMU_task-contrastChangeDe

In [9]:
import mne
path = '/mnt/dataset2/Datasets/HBN-EEG/ds005505-download/sub-NDARAC904DMU/eeg//sub-NDARAC904DMU_task-contrastChangeDetection_run-1_eeg.set'

In [13]:
test = mne.io.read_raw_eeglab(path)

/tmp/ipykernel_9091/2792990184.py:1: RuntimeWarning: Data will be preloaded. preload=False or a string preload is not supported when the data is stored in the .set file
  test = mne.io.read_raw_eeglab(path)


In [14]:
test._data.shape

(129, 117052)

## Multi-Dataset Report

In [9]:
# Base directory containing all datasets
base_dir = "/mnt/dataset2/Processed_datasets/EEG_Bench"

In [14]:
base_path = Path(base_dir)
datasets = sorted([d for d in base_path.iterdir() if d.is_dir()])
print(datasets)
results = []


for dataset_dir in datasets:
    h5_files = list(dataset_dir.glob("sub_*.h5"))
    if not h5_files:
        continue
    stats = check_dataset_unit(str(dataset_dir))
    stats["path"] = dataset_dir.name
    results.append(stats)

# Print summary table
print(f"{'Dataset':<20} {'Unit':<6} {'Max Amp':<12} {'Labels':<8} {'Categories':<10}")
print("-" * 70)

for r in results:
    if "error" in r:
        print(f"{r.get('path', 'Unknown'):<20} ERROR")
        continue
    unit_ok = "✓" if r['detected_unit'] == 'µV' else "✗"
    labels_ok = "✓" if r['num_labels'] and r['num_labels'] > 0 else "✗"
    cats_ok = "✓" if r['category_list'] and len(r['category_list']) > 0 else "✗"
    print(f"{r['path']:<20} {r['detected_unit']:<6} {r['max_amplitude']:<12.2e} {labels_ok:<8} {cats_ok:<10}")

print()
needs_update = [r['path'] for r in results if 'error' not in r and r['detected_unit'] != 'µV']
if needs_update:
    print(f"Datasets needing unit update: {needs_update}")
else:
    print("All datasets are in µV ✓")

[PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/AD65'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/BCIC2A'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/CineBrain'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/Daily_Movie'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/EmoEEG-MC_emo'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/FACED_emo'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/HFO'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/SEED'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/SEEDIV'), PosixPath('/mnt/dataset2/Processed_datasets/EEG_Bench/SEEDV')]


ValueError: invalid literal for int() with base 10: 'None'

## Batch Loading Test

In [ ]:
loader = load_dataset(data_dir, batch_size=32, num_workers=0)
print(f"Total samples: {len(loader.dataset)}")

for i, batch in enumerate(loader):
    if i >= 3:
        break
    unit, max_amp = detect_unit(batch['data'].numpy())
    print(f"Batch {i}: shape={batch['data'].shape}, unit={unit}, max_amp={max_amp:.2e}, labels={batch['label'].squeeze().tolist()[:5]}...")

## HDF5 Metadata Inspection

In [ ]:
h5_files = sorted(Path(data_dir).glob("sub_*.h5"))
if h5_files:
    with HDF5Reader(str(h5_files[0])) as reader:
        subj = reader.subject_attrs
        print(f"Dataset name: {subj.dataset_name}")
        print(f"Task type: {subj.task_type}")
        print(f"Downstream task: {subj.downstream_task_type}")
        print(f"Subject ID: {subj.subject_id}")
        print(f"Sampling rate: {subj.rsFreq} Hz")
        print(f"Montage: {subj.montage}")
        print(f"Channels ({len(subj.chn_name)}): {subj.chn_name[:5]}...")
        print(f"Num labels: {getattr(subj, 'num_labels', 'N/A')}")
        print(f"Categories: {getattr(subj, 'category_list', 'N/A')}")
        print(f"Num trials: {len(reader.get_trial_names())}")
        print(f"Num segments: {len(reader)}")